# Support Vector Machines: A Sparse Kernel Machine Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/svm_sparse_kernel_machines.ipynb)

Companion notebook for the [sesen.ai blog post](https://sesen.ai/blog/support-vector-machines-sparse-kernel-machines).

We fit linear and RBF support vector machines, verify the dual quadratic program from scratch with `cvxopt`, and explore how `C` and `gamma` shape the boundary.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

rng = np.random.default_rng(0)
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})
CLASS_COLORS = {-1: '#1f77b4', 1: '#d62728'}

def scatter(ax, X, y):
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c=CLASS_COLORS[1], marker='x', s=42, label='class +1')
    ax.scatter(X[y == -1, 0], X[y == -1, 1], c=CLASS_COLORS[-1], marker='o', s=36,
               edgecolors='white', linewidth=0.6, label='class -1')

def draw_boundary(ax, clf, X, pad=0.6, n=300):
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, n), np.linspace(y_min, y_max, n))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, np.sign(Z), levels=[-2, 0, 2], colors=['#aac8e6', '#f0b0b0'], alpha=0.25)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='black', linestyles=['--', '-', '--'])

def circle_svs(ax, clf):
    sv = clf.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=160, facecolors='none', edgecolors='#2ca02c',
               linewidths=1.6, label=f'support vectors (n={len(sv)})')

## 2. Quick Win: Linear SVM on Separable Blobs

On a clean linearly separable problem, the SVM's job is to find the unique hyperplane with the widest margin. Only the points on the margin boundary become support vectors.

In [ ]:
X_lin, y_lin = make_blobs(n_samples=40, centers=2, cluster_std=0.9,
                          center_box=(-2.5, 2.5), random_state=1)
y_lin = np.where(y_lin == 0, -1, 1)

linear = SVC(kernel='linear', C=10.0).fit(X_lin, y_lin)
print(f'support vectors: {len(linear.support_vectors_)} of {len(y_lin)}')

fig, ax = plt.subplots(figsize=(7, 5))
draw_boundary(ax, linear, X_lin)
scatter(ax, X_lin, y_lin)
circle_svs(ax, linear)
ax.set_title('Linear SVM: maximum-margin hyperplane')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(loc='lower right', fontsize=9)
plt.show()

## 3. RBF Kernel SVM on the Moons

When classes interleave non-linearly, swap the kernel. The boundary in input space curves, but it is still a flat hyperplane in the feature space the RBF kernel implicitly defines.

In [ ]:
X_m, y_m = make_moons(n_samples=200, noise=0.22, random_state=3)
X_m = StandardScaler().fit_transform(X_m)
y_m = np.where(y_m == 0, -1, 1)

rbf = SVC(kernel='rbf', C=4.0, gamma=1.5).fit(X_m, y_m)
print(f'RBF support vectors: {len(rbf.support_vectors_)} of {len(y_m)}')

fig, ax = plt.subplots(figsize=(7, 5))
draw_boundary(ax, rbf, X_m)
scatter(ax, X_m, y_m)
circle_svs(ax, rbf)
ax.set_title('RBF SVM on moons')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(loc='upper right', fontsize=9)
plt.show()

## 4. The Dual QP from Scratch with cvxopt

The hard-margin SVM dual is a quadratic program in the Lagrange multipliers `a`. We build it directly and check it matches scikit-learn's `SVC` to four decimal places.

**Note for Colab users:** uncomment the `pip install cvxopt` line on first run.

In [ ]:
# !pip install -q cvxopt
from cvxopt import matrix, solvers
solvers.options['show_progress'] = False

K = X_lin @ X_lin.T  # linear kernel matrix
N = len(y_lin)
P = matrix((y_lin[:, None] * y_lin[None, :]) * K)
q = matrix(-np.ones(N))
G = matrix(-np.eye(N))
h = matrix(np.zeros(N))
A = matrix(y_lin.astype(float).reshape(1, -1))
b_eq = matrix(0.0)

sol = solvers.qp(P, q, G, h, A, b_eq)
a = np.ravel(sol['x'])
sv_mask = a > 1e-5

w_dual = ((a * y_lin) @ X_lin)
b_dual = float(np.mean(y_lin[sv_mask] - X_lin[sv_mask] @ w_dual))

sklearn_clf = SVC(kernel='linear', C=1e6).fit(X_lin, y_lin)
print(f'cvxopt:  {sv_mask.sum()} SVs  w={w_dual.round(3)}  b={b_dual:.3f}')
print(f'sklearn: {sklearn_clf.n_support_.sum()} SVs  w={sklearn_clf.coef_.ravel().round(3)}  b={sklearn_clf.intercept_[0]:.3f}')

## 5. Soft-Margin C Sweep

On overlapping data the hard-margin formulation has no solution. The soft-margin SVM minimises
$$ C \sum_n \xi_n + \tfrac{1}{2}\|\mathbf{w}\|^2 $$
for slack variables $\xi_n \geq 0$. Small `C` allows wide margins with many violations; large `C` punishes violations into a narrower margin.

In [ ]:
X_o, y_o = make_blobs(n_samples=120, centers=2, cluster_std=0.95,
                       center_box=(-2, 2), random_state=4)
y_o = np.where(y_o == 0, -1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, C in zip(axes, [0.01, 1.0, 100.0]):
    clf = SVC(kernel='linear', C=C).fit(X_o, y_o)
    draw_boundary(ax, clf, X_o)
    scatter(ax, X_o, y_o)
    circle_svs(ax, clf)
    ax.set_title(f'C = {C}    SVs = {len(clf.support_vectors_)}')
    ax.set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
fig.suptitle('Soft-margin C sweep', y=1.02)
plt.show()

## 6. RBF Gamma Sweep

$\gamma$ controls the RBF length-scale. Small $\gamma$ underfits (smooth, washed-out boundary), large $\gamma$ overfits (boundary memorises individual points).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, g in zip(axes, [0.3, 1.5, 12.0]):
    clf = SVC(kernel='rbf', C=4.0, gamma=g).fit(X_m, y_m)
    draw_boundary(ax, clf, X_m)
    scatter(ax, X_m, y_m)
    circle_svs(ax, clf)
    ax.set_title(f'gamma = {g}    SVs = {len(clf.support_vectors_)}')
    ax.set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
fig.suptitle('RBF gamma sweep', y=1.02)
plt.show()

## 7. Hinge vs Cross-Entropy Loss

The hinge loss has a **flat region** for $z = yt > 1$. Confidently correct points contribute zero gradient, which is the algebraic source of SVM sparsity. Cross-entropy never goes flat, so logistic regression uses every point.

In [ ]:
z = np.linspace(-2.5, 2.5, 400)
hinge = np.maximum(0.0, 1.0 - z)
logistic = np.log1p(np.exp(-z)) / np.log(2.0)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(z, hinge, lw=2, label='hinge $[1-z]_+$ (SVM)', color='#1f77b4')
ax.plot(z, logistic, lw=2, label='cross-entropy (logistic)', color='#d62728')
ax.plot(z, (z < 0).astype(float), lw=1.4, ls='--', color='black', label='0/1 misclassification')
ax.axhline(0, color='gray', lw=0.6)
ax.axvline(0, color='gray', lw=0.6)
ax.set_xlabel('$z = y t$')
ax.set_ylabel('loss')
ax.set_title('Hinge vs cross-entropy: flat region drives sparsity')
ax.legend()
plt.show()

## 8. SVM vs Logistic Regression Side-by-Side

In [ ]:
svm = SVC(kernel='linear', C=1.0).fit(X_o, y_o)
lr = LogisticRegression(C=1.0, max_iter=2000).fit(X_o, y_o)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, clf, title in zip(axes, [svm, lr], ['Linear SVM (hinge)', 'Logistic regression (cross-entropy)']):
    draw_boundary(ax, clf, X_o)
    scatter(ax, X_o, y_o)
    if isinstance(clf, SVC):
        circle_svs(ax, clf)
    ax.set_title(title); ax.set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
axes[0].legend(loc='lower right', fontsize=8)
plt.show()

## Exercises

1. **Polynomial kernel.** Refit the moons with `kernel='poly', degree=3, C=4.0`. How does the support set count compare to the RBF kernel?
2. **Class imbalance.** Re-create `X_m, y_m` with a 90/10 class ratio (use `make_classification` with `weights=[0.9, 0.1]`) and refit the SVM with and without `class_weight='balanced'`. Inspect the change in the boundary and the SV count.
3. **Probability calibration.** Set `probability=True` on the moons RBF SVM and compare `predict_proba` outputs to a `LogisticRegression` fit on the same data. Plot a reliability diagram with `sklearn.calibration.calibration_curve`.
4. **Multi-class.** Generate a 3-class blob and fit `SVC(decision_function_shape='ovr')` and `SVC(decision_function_shape='ovo')`. Plot the decision regions and compare boundaries on the boundary-disagreement zones.